<a href="https://colab.research.google.com/github/FurkanAlpGurakan/AYM-Karar-Tahmini-NLP/blob/main/TF_IDF_Model_E%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Başlangıç Ayarları

Bu bölümde, veri işleme, modelleme ve görselleştirme için gerekli kütüphaneler yüklenir.
Google Colab üzerinde çalışıldığı için Google Drive bağlantısı kurulmuş ve eğitim verisinin bulunduğu dosya yolu tanımlanmıştır.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import re
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Google Drive bağlanıyor...")
    drive.mount('/content/drive')

ANA_KLASOR = "/content/drive/MyDrive/Dava_Sonuc_Tahmin"
VERI_YOLU = os.path.join(ANA_KLASOR, "Final_Veriseti_Egitim.xlsx")

print("TF-IDF + LinearSVC MODEL - GRAFİKLİ RAPORLAMA")
print("="*70)

TF-IDF + LinearSVC MODEL - GRAFİKLİ RAPORLAMA


Akıllı Metin Temizleme

Bu bölümde, hukuki karar metinleri için özel bir ön işleme uygulanmaktadır.
Anlamsız ve sık geçen kelimeler çıkarılırken, karar sonucunu etkileyen önemli hukuki terimler özellikle korunmaktadır.
Sayısal ifadeler genelleştirilerek metinler modele daha uygun hale getirilmektedir.

In [ ]:
MINIMAL_STOPWORDS = set([
    "bir", "ve", "veya", "ile", "bu", "şu", "o", "olan", "olarak",
    "için", "gibi", "göre", "kadar", "daha", "en", "çok", "az"
])

ONEMLI_TERIMLER = set([
    "kabul", "ret", "reddi", "kabulü", "beraat", "mahkumiyet",
    "temyiz", "bozma", "onama", "hüküm", "itiraz", "düzeltme",
    "tazminat", "ceza", "hapis", "para", "kusur", "sorumluluk",
    "ihlal", "edilemez"
])

def akilli_metin_temizle(metin: str) -> str:
    """Hukuki terimleri koruyarak metin temizleme."""
    if pd.isna(metin):
        return ""

    metin = str(metin).lower()
    metin = re.sub(r'\b\d{1,2}\b', ' iki_haneli_sayi ', metin)
    metin = re.sub(r'\b\d{3,4}\b', ' yil_benzeri ', metin)
    metin = re.sub(r'\b\d{5,}\b', ' buyuk_sayi ', metin)
    metin = re.sub(r'[^a-zçğıöşü\s]', ' ', metin)
    metin = re.sub(r'\s+', ' ', metin).strip()

    kelimeler = metin.split()
    temiz_kelimeler = []

    for kelime in kelimeler:
        if len(kelime) <= 2:
            continue
        if kelime in ONEMLI_TERIMLER:
            temiz_kelimeler.append(kelime)
            continue
        if kelime in MINIMAL_STOPWORDS:
            continue
        temiz_kelimeler.append(kelime)

    return " ".join(temiz_kelimeler)

Veri Yükleme ve Hazırlık

Bu bölümde, eğitim verisinin dosya yolu kontrol edilmekte ve Excel dosyası yüklenmektedir.
Eksik metin veya etiket içeren kayıtlar temizlenerek veri seti hazır hale getirilir.
Metinler ön işleme tabi tutulduktan sonra veri, sınıf dağılımı korunacak şekilde eğitim ve test kümelerine ayrılmaktadır.

In [ ]:
if not os.path.exists(VERI_YOLU):
    print(f" HATA: '{VERI_YOLU}' dosyası bulunamadı!")
else:
    print("\n Veri yükleniyor...")
    df = pd.read_excel(VERI_YOLU)
    df = df.dropna(subset=["Metin", "Etiket"])

    print(f" Veri Seti Yüklendi. Toplam Karar: {len(df)}")
    print("-" * 50)
    print(" Sınıf Dağılımı:\n", df["Etiket"].value_counts())

    print("\n Metinler temizleniyor...")
    df["Metin_Temiz"] = df["Metin"].apply(akilli_metin_temizle)

    # Veriyi Ayır (%80 Eğitim - %20 Test)
    X_train, X_test, y_train, y_test = train_test_split(
        df["Metin_Temiz"],
        df["Etiket"],
        test_size=0.2,
        random_state=42,
        stratify=df["Etiket"]
    )

    print("-" * 50)
    print(f" Eğitim Verisi: {len(X_train)} adet")
    print(f" Test Verisi:   {len(X_test)} adet")


 Veri yükleniyor...
 Veri Seti Yüklendi. Toplam Karar: 1800
--------------------------------------------------
 Sınıf Dağılımı:
 Etiket
Kabul_Edilemez    600
İhlal_Yok        600
İhlal_Var        600
Name: count, dtype: int64

 Metinler temizleniyor...
--------------------------------------------------
 Eğitim Verisi: 1440 adet
 Test Verisi:   360 adet


Model Tanımı ve Doğrulama

Bu bölümde, TF-IDF vektörleştirme ve LinearSVC sınıflandırıcısından oluşan model pipeline’ı tanımlanmaktadır.
Modelin genellenebilirliği, sınıf dağılımı korunarak 10-Fold Cross-Validation ile değerlendirilmektedir.



In [ ]:
print(" MODEL TANIMI: TF-IDF + LinearSVC")

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        C=2.0,
        dual="auto",
        max_iter=2000,
        random_state=42
    ))
])

# Cross-Validation
print("\n 10-Fold Cross-Validation yapılıyor (Lütfen bekleyin)...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

print("-" * 50)
print(f" Cross-Validation Skorları: {[f'%{s*100:.2f}' for s in cv_scores]}")
print(f" ORTALAMA CROSS-VALİDATİON BAŞARISI: %{cv_scores.mean()*100:.2f} (±{cv_scores.std()*100:.2f})")

 MODEL TANIMI: TF-IDF + LinearSVC

 10-Fold Cross-Validation yapılıyor (Lütfen bekleyin)...


KeyboardInterrupt: 

Model Eğitimi ve Test

Bu bölümde, model tüm eğitim verisi kullanılarak eğitilmektedir.
Eğitim tamamlandıktan sonra test verisi üzerinde tahmin alınmakta ve doğruluk oranı ile detaylı sınıflandırma metrikleri raporlanmaktadır.

In [ ]:
print("\n Model tüm eğitim setiyle eğitiliyor...")
model.fit(X_train, y_train)
print(" Eğitim tamamlandı!")

# Test ve Değerlendirme
tahminler = model.predict(X_test)
basari = accuracy_score(y_test, tahminler)
basari_yuzdesi = basari * 100

print("\n" + "="*50)
print(f" TEST DOĞRULUĞU: %{basari_yuzdesi:.2f}")
print("="*50)

print("\n DETAYLI RAPOR:")
print(classification_report(y_test, tahminler))

Metrikler ve Grafikler

Bu bölümde, test sonuçlarından sınıf bazlı precision/recall/F1 değerleri ve confusion matrix hesaplanmaktadır.
Ardından cross-validation skorları, sınıf bazlı performans, confusion matrix, test/cross-validation karşılaştırması ve F1-score dağılımını gösteren grafikler oluşturularak model performansı görsel olarak sunulmaktadır.

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_test, tahminler)
labels = sorted(df["Etiket"].unique())
cm = confusion_matrix(y_test, tahminler)

print("\n Grafikler oluşturuluyor...")
fig = plt.figure(figsize=(18, 10))

# Grafik 1: Cross-Validation
ax1 = plt.subplot(2, 3, 1)
ax1.plot(range(1, 11), cv_scores * 100, marker='o', color='#4dabf7')
ax1.axhline(y=cv_scores.mean()*100, color='green', linestyle='--', label='Ortalama')
ax1.set_title('Cross-Validation Skorları', fontweight='bold')
ax1.set_ylim([75, 95])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Grafik 2: Sınıf Bazlı Performans
ax2 = plt.subplot(2, 3, 2)
x = np.arange(len(labels))
width = 0.25
ax2.bar(x - width, precision*100, width, label='Precision', color='#ffa94d')
ax2.bar(x, recall*100, width, label='Recall', color='#4dabf7')
ax2.bar(x + width, f1*100, width, label='F1-Score', color='#6bcf7f')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=15)
ax2.set_title('Sınıf Bazlı Metrikler', fontweight='bold')
ax2.legend()
ax2.set_ylim([70, 100])

# Grafik 3: Confusion Matrix
ax3 = plt.subplot(2, 3, 3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax3)
ax3.set_title(f'Confusion Matrix\nDoğruluk: %{basari_yuzdesi:.2f}', fontweight='bold')
ax3.set_ylabel('Gerçek')
ax3.set_xlabel('Tahmin')

# Grafik 4: Test vs CV
ax4 = plt.subplot(2, 3, 4)
comps = {'Test': basari_yuzdesi, 'CV Ort': cv_scores.mean()*100}
ax4.bar(comps.keys(), comps.values(), color=['#4dabf7', '#6bcf7f'])
ax4.set_ylim([75, 95])
ax4.set_title('Test vs CV Kıyaslaması', fontweight='bold')
for i, v in enumerate(comps.values()):
    ax4.text(i, v, f'%{v:.2f}', ha='center', va='bottom', fontweight='bold')

# Grafik 5: F1-Score Yatay Bar
ax5 = plt.subplot(2, 3, 5)
colors = ['#6bcf7f' if s >= 0.85 else '#ff6b6b' for s in f1]
ax5.barh(labels, f1*100, color=colors)
ax5.set_xlim([70, 100])
ax5.set_title('F1-Score Sıralaması', fontweight='bold')
ax5.axvline(x=85, color='red', linestyle='--')

plt.tight_layout()
plt.show()

Model ve Çıktıların Kaydedilmesi

Bu bölümde, eğitilen model için Google Drive üzerinde bir klasör oluşturulmakta ve model dosyası performans değerlerini içerecek şekilde adlandırılarak kaydedilmektedir.
Ayrıca test sonuçlarını içeren özet bir performans raporu metin dosyası olarak oluşturulmakta ve tüm işlem süreci tamamlanmaktadır.

In [ ]:
KLASOR_ADI = "TF-IDF_Model_Final"
MODEL_KLASORU = os.path.join(ANA_KLASOR, KLASOR_ADI)

if not os.path.exists(MODEL_KLASORU):
    os.makedirs(MODEL_KLASORU)
    print(f" Klasör oluşturuldu: {MODEL_KLASORU}")

# İsimlendirme
dosya_ismi = f"TF-IDF_LinearSVC_ACC_{basari_yuzdesi:.2f}_CV_{cv_scores.mean()*100:.2f}.joblib"
kayit_yolu = os.path.join(MODEL_KLASORU, dosya_ismi)

print(f"\n Model kaydediliyor...")
joblib.dump(model, kayit_yolu)

# Raporu TXT olarak yaz
rapor_yolu = os.path.join(MODEL_KLASORU, f"Model_Raporu_ACC_{basari_yuzdesi:.2f}.txt")
with open(rapor_yolu, 'w', encoding='utf-8') as f:
    f.write(f"MODEL RAPORU\n{'='*20}\n")
    f.write(f"Test Accuracy: %{basari_yuzdesi:.2f}\n")
    f.write(f"CV Average: %{cv_scores.mean()*100:.2f}\n\n")
    f.write(classification_report(y_test, tahminler))

print(f" Model: {kayit_yolu}")
print(f" Rapor: {rapor_yolu}")
print("\n İŞLEM TAMAMLANDI!")

ARAYÜZ


 Kütüphaneler ve Drive bağlantısı

Bu bölümde veri işleme, modelleme ve değerlendirme için gerekli kütüphaneler yüklenir. Google Drive bağlantısı kurularak eğitim verisinin bulunduğu dosyaya erişim sağlanır.

In [ ]:
import sys
!{sys.executable} -m pip install python-docx gradio -q

import gradio as gr
import docx
import joblib
import re
import os
from google.colab import drive

# DRIVE BAĞLANTISI VE MODEL YOLU
if not os.path.exists('/content/drive'):
    print(" Google Drive bağlanıyor...")
    drive.mount('/content/drive')

ANA_KLASOR = "/content/drive/MyDrive/Dava_Sonuc_Tahmin"
MODEL_KLASORU = os.path.join(ANA_KLASOR, "TF-IDF_Model_Final")

print(f" Model Klasörü Kontrol Ediliyor: {MODEL_KLASORU}")


En Güncel Modelin Yüklenmesi

Bu bölümde, belirtilen model klasörü içerisindeki .joblib uzantılı dosyalar taranır ve en son değiştirilen model otomatik olarak seçilerek yüklenir.
Amaç, manuel seçim yapmadan her zaman en güncel ve son eğitilmiş modeli kullanmaktır.

In [ ]:
def son_modeli_yukle(model_klasoru: str):
    """Klasördeki .joblib dosyaları içinden EN SON değiştirilen modeli yükler."""
    if not os.path.exists(model_klasoru):
        print(" HATA: Model klasörü bulunamadı!")
        return None, None

    dosyalar = os.listdir(model_klasoru)
    model_dosyalari = [f for f in dosyalar if f.endswith('.joblib')]

    if not model_dosyalari:
        print(" HATA: Klasörde .joblib dosyası bulunamadı!")
        return None, None

    en_son_model = max(
        model_dosyalari,
        key=lambda f: os.path.getmtime(os.path.join(model_klasoru, f))
    )
    model_yolu = os.path.join(model_klasoru, en_son_model)

    print(f" Model Bulundu: {en_son_model}")
    print(f" Model yükleniyor...")

    try:
        yuklu_model = joblib.load(model_yolu)
        print(f" Model başarıyla yüklendi!")
        return yuklu_model, model_yolu
    except Exception as e:
        print(f" KRİTİK HATA: Model yüklenemedi.\nHata Detayı: {e}")
        return None, None


model, MODEL_YOLU = son_modeli_yukle(MODEL_KLASORU)


Metin Temizleme ve Etiket Tanımları

Bu bölümde, metin ön işleme sırasında kullanılacak temel stopword listesi ve karar sonucunu etkileyen önemli hukuki terimler tanımlanır.
Ayrıca model çıktılarının kullanıcıya daha anlaşılır şekilde sunulabilmesi için sınıf etiketlerinin açıklamaları belirlenir.

In [ ]:
MINIMAL_STOPWORDS = set([
    "bir", "ve", "veya", "ile", "bu", "şu", "o", "olan", "olarak",
    "için", "gibi", "göre", "kadar", "daha", "en", "çok", "az"
])

ONEMLI_TERIMLER = set([
    "kabul", "ret", "reddi", "kabulü", "beraat", "mahkumiyet",
    "temyiz", "bozma", "onama", "hüküm", "itiraz", "düzeltme",
    "tazminat", "ceza", "hapis", "para", "kusur", "sorumluluk",
    "ihlal", "edilemez"
])

ETIKET_ACIKLAMA = {
    "Kabul_Edilemez": "Başvuru Kabul Edilemez",
    "İhlal_Yok": "İhlal Yok",
    "İhlal_Var": "İhlal Var"
}


Akıllı Metin Temizleme ve Karar Kısmının Ayrıştırılması

Bu bölümde, hukuki metinler eğitim aşamasındaki ön işleme ile tam uyumlu olacak şekilde temizlenir.
Sayısal ifadeler genelleştirilir, gereksiz karakterler kaldırılır ve karar sonucunu ele verebilecek bölümler metinden ayrıştırılmaya çalışılır.

In [ ]:
def akilli_metin_temizle(metin: str) -> str:
    """Hukuki terimleri koruyarak metin temizleme (eğitimdeki ile tutarlı)."""
    if not metin:
        return ""

    metin = str(metin).lower()

    metin = re.sub(r'\b\d{1,2}\b', ' iki_haneli_sayi ', metin)
    metin = re.sub(r'\b\d{3,4}\b', ' yil_benzeri ', metin)
    metin = re.sub(r'\b\d{5,}\b', ' buyuk_sayi ', metin)

    metin = re.sub(r'[^a-zçğıöşü\s]', ' ', metin)
    metin = re.sub(r'\s+', ' ', metin).strip()

    kelimeler = metin.split()
    temiz_kelimeler = []

    for kelime in kelimeler:
        if len(kelime) <= 2:
            continue
        if kelime in ONEMLI_TERIMLER:
            temiz_kelimeler.append(kelime)
            continue
        if kelime in MINIMAL_STOPWORDS:
            continue
        temiz_kelimeler.append(kelime)

    return " ".join(temiz_kelimeler)


def metni_kes(metin: str):
    """Sonuç/gerekçe gibi karar kısmını ele veren bölümleri kesmeye çalışır."""
    kesme_noktalari = [
        "Açıklanan gerekçelerle",
        "Bu gerekçelerle",
        "HÜKÜM",
        "SONUÇ",
        "YARGILAMA GİDERLERİ",
        "KARAR",
        "karar verilmiştir"
    ]

    en_iyi_indeks = len(metin)
    kesen_kelime = None

    for nokta in kesme_noktalari:
        indeks = metin.lower().find(nokta.lower())
        if indeks != -1 and indeks > 500:
            if indeks < en_iyi_indeks:
                en_iyi_indeks = indeks
                kesen_kelime = nokta

    if kesen_kelime:
        return metin[:en_iyi_indeks], f" '{kesen_kelime}' kelimesinden sonrası kesildi."
    else:
        return metin, " Hiçbir kesme noktası bulunamadı."


Dosyadan Metin Okuma

Bu bölümde, kullanıcı tarafından yüklenen .docx veya .txt formatındaki dosyalardan metin güvenli şekilde okunur.
Desteklenmeyen dosya türleri veya okuma sırasında oluşan hatalar kullanıcıya uygun mesajlarla bildirilir.

In [ ]:
def dosyadan_metin_oku(file_obj):
    """DOCX veya TXT dosyasından metin oku."""
    try:
        if file_obj.name.endswith('.docx'):
            doc = docx.Document(file_obj.name)
            full_text = [p.text for p in doc.paragraphs if p.text.strip()]
            ham_metin = " ".join(full_text)
        elif file_obj.name.endswith('.txt'):
            with open(file_obj.name, 'r', encoding='utf-8') as f:
                ham_metin = f.read()
        else:
            return None, " Desteklenmeyen dosya formatı! Sadece .docx veya .txt desteklenmektedir."

        return ham_metin, None
    except Exception as e:
        return None, f" Dosya okuma hatası: {str(e)}"


Ana Analiz ve Tahmin Fonksiyonu

Bu bölümde kullanıcıdan gelen metin (dosya veya direkt giriş) alınır, kısa metin kontrolleri yapılır ve metin ön işleme uygulanır.
Ardından model ile tahmin üretilir, decision skorlarından göreli güven oranı hesaplanır ve sonuç; metin istatistikleriyle birlikte rapor formatında kullanıcıya döndürülür.

In [ ]:
def analiz_et(file_obj, direkt_metin):
    """Dosya veya direkt metin ile tahmin yap."""

    if model is None:
        return " Model yüklenemedi! Lütfen model yolunu ve klasörü kontrol edin.", "", ""

    ham_metin = None

    if direkt_metin and str(direkt_metin).strip():
        ham_metin = str(direkt_metin)
        metin_kaynagi = " Direkt metin kullanıldı"
    elif file_obj is not None:
        ham_metin, hata = dosyadan_metin_oku(file_obj)
        if hata:
            return hata, "", ""
        metin_kaynagi = f" Dosya: {os.path.basename(file_obj.name)}"
    else:
        return " Lütfen bir dosya yükleyin veya metin girin!", "", ""

    if len(ham_metin.strip()) < 50:
        return " Metin çok kısa görünüyor. Lütfen daha uzun bir karar/gerekçe metni sağlayın.", "", ""

    try:
        kesilmis_metin, kesme_bilgisi = metni_kes(ham_metin)
        temiz_metin = akilli_metin_temizle(kesilmis_metin)

        if not temiz_metin.strip():
            return " Temizleme işleminden sonra metin boş kaldı. Metni kontrol edin.", "", ""

        kelime_sayisi = len(temiz_metin.split())
        karakter_sayisi = len(temiz_metin)

        tahmin = model.predict([temiz_metin])[0]
        tahmin_okunur = ETIKET_ACIKLAMA.get(tahmin, tahmin)

        guven = 0.0
        olasiliklar = "Olasılık skorları hesaplanamadı."

        try:
            import numpy as np

            if hasattr(model.named_steps['clf'], 'decision_function'):
                decision_scores = model.decision_function([temiz_metin])[0]
                siniflar = list(model.classes_)  # sınıf sırası korunur

                exp_scores = np.exp(decision_scores - np.max(decision_scores))
                probs = exp_scores / exp_scores.sum()

                if tahmin in siniflar:
                    tahmin_index = siniflar.index(tahmin)
                    guven = probs[tahmin_index] * 100.0

                olasilik_satirlari = []
                for cls, p in zip(siniflar, probs):
                    aciklama = ETIKET_ACIKLAMA.get(cls, cls)
                    olasilik_satirlari.append(f"   • {aciklama} ({cls}): %{p*100:.2f}")
                olasiliklar = "\n".join(olasilik_satirlari)

        except Exception:
            olasiliklar = "Olasılık skorları hesaplanırken hata oluştu."
            guven = 0.0

        rapor = f"""

 TAHMİN SONUCU: {tahmin_okunur} ({tahmin})
{" Güven Oranı: %" + f"{guven:.2f}" if guven > 0 else ""}

 OLASLIK DAĞILIMI:
{olasiliklar}

 METİN İSTATİSTİKLERİ:
   • Toplam Kelime: {kelime_sayisi:,}
   • Toplam Karakter: {karakter_sayisi:,}
   • {metin_kaynagi}
   • {kesme_bilgisi}

⚠️ DİKKAT: Bu sistem bir karar destek aracıdır,
           çıktıların hukuki bağlayıcılığı yoktur.
"""

        gosterilecek_metin = temiz_metin[:2000]
        if len(temiz_metin) > 2000:
            gosterilecek_metin += f"\n\n... (Toplam {kelime_sayisi} kelime, ilk 2000 karakter gösteriliyor)"

        ham_onizleme = ham_metin[:1000] + "..." if len(ham_metin) > 1000 else ham_metin

        return rapor, gosterilecek_metin, ham_onizleme

    except Exception as e:
        import traceback
        hata_detay = traceback.format_exc()
        return f" Analiz Hatası:\n{str(e)}\n\nDetay:\n{hata_detay}", "", ""


Gradio Arayüzünün Oluşturulması

Bu bölümde kullanıcı etkileşimi için Gradio tabanlı bir arayüz oluşturulur.
Kullanıcı ister dosya yükleyerek ister metni doğrudan girerek analiz başlatabilir; sonuç raporu, işlenmiş metin ve ham metin ayrı alanlarda görüntülenir.

In [ ]:
def arayuz_olustur():
    """Gradio arayüzünü oluşturur."""
    try:
        tema = gr.themes.Soft(
            primary_hue="blue",
            secondary_hue="gray",
        )

        with gr.Blocks(theme=tema, title="AYM Karar Tahmin Sistemi") as demo:

            gr.Markdown("""
            #  Anayasa Mahkemesi Karar Tahmin Sistemi
            ### TF-IDF + LinearSVC Model ile Dava Sonucu Tahmini
            ---
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("###  Giriş Seçenekleri")

                    dosya_input = gr.File(
                        label="1️ Dava Dosyası Yükle (.docx veya .txt)",
                        file_types=[".docx", ".txt"]
                    )

                    gr.Markdown("**VEYA**")

                    metin_input = gr.Textbox(
                        label="2️ Metni Direkt Yapıştır",
                        placeholder="Dava metnini buraya yapıştırın...",
                        lines=8
                    )

                    btn = gr.Button(" Analiz Et", variant="primary")

                with gr.Column(scale=1):
                    gr.Markdown("###  Analiz Sonuçları")
                    sonuc_output = gr.Textbox(label=" Tahmin Raporu", lines=20, max_lines=25)

            with gr.Tabs():
                with gr.Tab(" İşlenmiş Metin"):
                    temiz_metin_output = gr.Textbox(label="Model Tarafından İşlenen Metin", lines=10, max_lines=15)

                with gr.Tab(" Ham Metin"):
                    ham_metin_output = gr.Textbox(label="Orijinal Metin (Önizleme)", lines=10, max_lines=15)

            gr.Markdown("###  Örnek Kullanım")
            gr.Examples(
                examples=[
                    ["Başvurucu, hakkaniyete uygun yargılanma hakkının ihlal edildiğini iddia etmiştir. Mahkeme incelemesi sonucunda başvurucunun iddialarının yerinde olduğu tespit edilmiştir."],
                    ["Başvurucunun iddiaları incelenmiş ve yerinde bulunmamıştır. Başvurunun açıkça dayanaktan yoksun olduğu anlaşılmıştır."]
                ],
                inputs=metin_input,
                label="Örnek Metinler"
            )

            btn.click(
                fn=analiz_et,
                inputs=[dosya_input, metin_input],
                outputs=[sonuc_output, temiz_metin_output, ham_metin_output]
            )

            gr.Markdown(f"""
            ---
            **📁 Kullanılan Model Dosyası:** `{os.path.basename(MODEL_YOLU) if MODEL_YOLU else "Yüklenemedi"}`
            """)

        return demo

    except Exception as e:
        print(f" Arayüz oluşturma hatası: {e}")
        import traceback
        traceback.print_exc()
        return None


Uygulamanın Başlatılması ve Kontroller

Bu bölümde modelin başarıyla yüklenip yüklenmediği kontrol edilir.
Model mevcutsa Gradio arayüzü başlatılır; herhangi bir hata durumunda kullanıcıya gerekli kontrolleri yapması için bilgilendirici mesajlar gösterilir.

In [ ]:
if model is not None:
    print("\n" + "="*70)
    print(" Gradio Arayüzü Başlatılıyor...")
    print("="*70)

    demo = arayuz_olustur()

    if demo is not None:
        demo.launch(
            share=True,
            debug=True,
            show_error=True,
            inline=False
        )
    else:
        print(" Arayüz oluşturulamadı!")
else:
    print("\n" + "="*70)
    print(" MODEL YÜKLENEMEDİĞİ İÇİN ARAYÜZ BAŞLATILAMADI!")
    print("="*70)
    print("\nLütfen şunları kontrol edin:")
    print("1. Model klasörü doğru mu?")
    print("2. Google Drive bağlantısı var mı?")
    print("3. Model dosyası .joblib formatında mı?")
    print(f"\nAranan klasör: {MODEL_KLASORU}")



 Gradio Arayüzü Başlatılıyor...


/tmp/ipython-input-939636077.py:9: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=tema, title="AYM Karar Tahmin Sistemi") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9801a70db1172540b4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
